In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [11]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate, ClassicalRegister
import math 

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

backend = BasicSimulator()

# Create the entanglement
# Also need a register for attacker to store intermediate value in.
def create_initial_state():
    q = QuantumCircuit(2)
    attacker_reg = ClassicalRegister(1)
    q.add_register(attacker_reg)
    q.x(1)
    q.h(0) 
    q.cx(0, 1) 
    q.z(0) 
    return q, attacker_reg

# Attacker functionality
# Need to measure the qubit, store it, then make it back into its original state.
def attacker(q, attacker_reg):
    q.measure(1, attacker_reg[0])
    q.reset(1)
    q.x(1).c_if(attacker_reg, 1)
    return q

# Generate three outcomes with 1/3 probability each.
# Rotate qubit by the angle so the probability of measuring 0 is 1/3.
# Then do a 50/50 with a H gate on the remaining 2/3 probability.
def random():
    q_1 = QuantumCircuit(1, 1)
    angle = math.acos(math.sqrt(1/3)) * 2
    q_1.ry(angle, 0)
    q_1.measure(0, 0) 
    compiled_1 = transpile(q_1, backend)
    job_1_sim = backend.run(compiled_1, shots=1)
    result_1_sim = job_1_sim.result()
    output_1 = list(result_1_sim.get_counts(compiled_1).keys())[0] 

    if output_1 == "0":
        return "1st_third"
    else:
        q_2 = QuantumCircuit(1, 1)
        q_2.h(0) 
        q_2.measure(0, 0) 
        compiled_2 = transpile(q_2, backend)
        job_2_sim = backend.run(compiled_2, shots=1)
        result_2_sim = job_2_sim.result() 
        output_2 = list(result_2_sim.get_counts(compiled_2).keys())[0] 
        if output_2 == "0":
            return "2nd_third"
        else:
            return "3rd_third"

# Build W and V
root2 = math.sqrt(2)
denom1 = math.sqrt(4 + 2*root2)
denom2 = math.sqrt(4 - 2*root2)
W_matrix = [ [ -1 / denom1 , (1 + root2) / denom1 ],  
             [  1 / denom2 , (root2 - 1) / denom2 ] ]
V_matrix = [ [  1 / denom1 , (1 + root2) / denom1 ],   
             [ -1 / denom2 , (root2 - 1) / denom2 ] ]

N = 100
N_rounds = int(9 * N / 2)
key = []
counts = {"XW": [], "XV": [], "ZW": [], "ZV": []} 

# For each iteration:
# Create an entanglement
# Have an attacker intercept this bit
# Alice chooses a bit using the 3-outcome random
# Bob does the same
# Measure the qubits
for i in range(N_rounds):
    q, attacker_reg = create_initial_state()
    q = attacker(q, attacker_reg)
    Ai = random()
    if Ai == "1st_third":
        q.h(0) 
    elif Ai == "2nd_third":
        q.unitary(W_matrix, [0])

    Bj = random()
    if Bj == "1st_third":
        q.unitary(W_matrix, [1])
    elif Bj == "3rd_third":
        q.unitary(V_matrix, [1])

    q.measure_all()

    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result()
    output = list(result_sim.get_counts(compiled).keys())[0]

    if output[1] == "0":
        a = 1
    else:
        a = -1
    if output[0] == "0":
        b = 1 
    else:
        b = -1
    # There are 2 outcomes out of the possible 9 which can be added to the key.
    if (Ai == "2nd_third" and Bj == "1st_third") or (Ai == "3rd_third" and Bj == "2nd_third"):
        if output[1] == "0":
            key.append(0)
        else:
            key.append(1)

    # Counts for bell test
    if Ai == "1st_third" and Bj == "1st_third":
        counts["XW"].append(a * b)
    elif Ai == "1st_third" and Bj == "3rd_third":
        counts["XV"].append(a * b)
    elif Ai == "3rd_third" and Bj == "1st_third":
        counts["ZW"].append(a * b)
    elif Ai == "3rd_third" and Bj == "3rd_third":
        counts["ZV"].append(a * b)

# Calculate bell value
# Should be lower because attacker broke the entanglement
result = abs(sum(counts["XW"])/len(counts["XW"]) - sum(counts["XV"])/len(counts["XV"]) + sum(counts["ZW"])/len(counts["ZW"]) + sum(counts["ZV"])/len(counts["ZV"]))

print("Key size: " + str(len(key)))
print("Bell value: " + str(result))


Key size: 91
Bell value: 1.1683621933621935
